# Day 5.3 — Tool Registry and Discovery
On Day 1 a tool was an `if name == "search":` branch. That works for one tool and one
agent. It stops working the moment two agents need *different* tools.

A registry stores each tool once — name, description, input schema, risk level and the
function — and hands out only the subset an agent is allowed to see.


## Before you begin

### Learning outcomes

- Register tools with a schema and a risk level, and list them.
- Give each agent a scoped view of the registry instead of everything.
- Reject malformed arguments before any function runs.

Architecture reference: [Day 5 diagrams D16](../diagrams/source/day_05.md).

### Expected observation

The research agent sees one tool while four are registered, and a draft with no body is refused before `create_draft` is ever called.


## Concept briefing

## Registry, validation and policy

A tool registry stores names, descriptions, input schemas, executors and local risk
classifications. Discovery answers "what capabilities are visible?" Validation answers
"are these arguments structurally acceptable?" Policy answers "may this agent execute
this action now?" These are separate decisions.

The runtime should fail closed on unknown tools, invalid arguments and disallowed actions.
It should never ask the same model that proposed an action to make the authoritative
permission decision.


In [ ]:
# --- Course setup: run this cell first -----------------------------------------
# 1) Locate this day's folder so we can import from src/ and read data/ no matter
#    where Jupyter, VS Code, or Colab started. Every file path below goes through
#    PROJECT_ROOT, never through the current working directory.
import os, sys
from pathlib import Path

def find_project_root(marker="src/mini_harness"):
    here = Path.cwd().resolve()
    for folder in [here, *here.parents]:
        for candidate in [folder, *folder.glob("day_*")]:
            if (candidate / marker).exists():
                return candidate
    raise FileNotFoundError(
        "Course folder not found. On Google Colab run the 'Colab bootstrap' cell at the top "
        "of the day notebook first; locally, start Jupyter inside the repository folder."
    )

PROJECT_ROOT = find_project_root()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

# 2) Load the API key from the .env file at the repository root (Day 1.1 shows how to
#    create it). If no key is present we stay in deterministic MOCK mode: every cell
#    still runs, answers are fixed strings, and no credit is spent.
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(usecwd=True))
LIVE = bool(os.getenv("OPENROUTER_API_KEY"))

print("Project root :", PROJECT_ROOT)
print("Mode         :", "LIVE (OpenRouter)" if LIVE else "MOCK (no OPENROUTER_API_KEY found)")

# 3) Day 5 helper: agent configurations are DATA. Read one from configs/<name>.json
#    and turn it into the AgentConfig dataclass the runtime expects.
import json
from mini_harness import AgentConfig, ModelConfig

def load_config(name):
    raw = json.loads((PROJECT_ROOT / "configs" / f"{name}.json").read_text(encoding="utf-8"))
    raw["model"] = ModelConfig(**raw["model"])   # nested dict -> nested dataclass
    return AgentConfig(**raw)

print("Configs      :", sorted(p.stem for p in (PROJECT_ROOT / "configs").glob("*.json")))

## Step 1 — What a registered tool actually carries

Four tools, deliberately spanning the four risk levels you will meet in Day 5.4.


In [ ]:
from mini_harness import build_demo_registry

registry = build_demo_registry()

print(f"{'tool':<16}{'risk':<13}{'required arguments'}")
print("-" * 60)
for spec in registry.discover():
    required = ", ".join(spec.input_schema.get("required", [])) or "(none)"
    print(f"{spec.name:<16}{spec.risk:<13}{required}")

print()
print("Full description sent to the model for one tool:")
print("  name       :", registry.get("create_draft").spec.name)
print("  description:", registry.get("create_draft").spec.description)
print("  schema     :", registry.get("create_draft").spec.input_schema)

## Step 2 — Discovery is scoped per agent

`discover(allowed)` is what gets turned into the tool list sent to the model. A tool the
agent may not use is never mentioned to it at all.


In [ ]:
research = load_config("research_agent")
task = load_config("task_agent")

print("Registered in total     :", [s.name for s in registry.discover()])
print("research_agent can see  :", [s.name for s in registry.discover(research.allowed_tools)])
print("task_agent can see      :", [s.name for s in registry.discover(task.allowed_tools)])
print()
print("erase_workspace is registered but appears in neither list.")
print("Not mentioning a tool is the cheapest safety control there is.")

## Step 3 — A valid call

`call()` validates first, then runs the function. Here everything is in order.


In [ ]:
result = registry.call("lookup_notes", {"query": "harness"})
print("Tool     :", "lookup_notes")
print("Arguments:", {"query": "harness"})
print("Returned :", result)

## Step 4 — Four ways to be rejected

Validation is structural: it checks the *shape* of the arguments. Each failure below
happens before the tool function is entered, so no side effect can occur.


In [ ]:
attempts = [
    ("create_draft", {"subject": "Body is missing"},          "a required argument is absent"),
    ("create_draft", {"subject": "x", "body": 42},            "an argument has the wrong type"),
    ("lookup_notes", {"query": "ok", "extra": "surprise"},    "an argument nobody declared"),
    ("no_such_tool", {"query": "ok"},                         "the tool does not exist"),
]

for name, arguments, why in attempts:
    try:
        registry.call(name, arguments)
    except Exception as exc:                  # noqa: BLE001 - we want to show the type
        print(f"{why:<32} -> {type(exc).__name__}: {exc}")
    else:
        print(f"{why:<32} -> ACCEPTED (this would be a bug)")

print()
print("None of the four tool functions ran. That is the point of validating first.")

## Step 5 — Being visible is not being allowed

This is the sentence to remember from Day 5. `send_email` is in the task agent's
allow-list and passes validation — and still does not get to run on its own.


In [ ]:
from mini_harness import decide

spec = registry.get("send_email").spec
print("Is send_email discoverable by task_agent?",
      spec.name in [s.name for s in registry.discover(task.allowed_tools)])
print("Do these arguments validate?", end=" ")
registry.validate("send_email", {"to": "a@b.test", "subject": "s", "body": "b"})
print("yes")
print("So may it run?  ->", decide(task, spec))
print()
print("Three separate questions:")
print("  discovery  : does this agent get told the tool exists?")
print("  validation : are these arguments structurally acceptable?")
print("  policy     : may this agent perform this action right now?  (Day 5.4)")

### Try it yourself

Register your own read-only tool with one required string argument, then prove that
passing a number instead of a string is refused.


In [ ]:
# --- Worked solution ---
from mini_harness import ToolRegistry, ToolSpec

mine = ToolRegistry()

# A ToolSpec is: name, description, JSON-Schema for the arguments, local risk level.
mine.register(
    ToolSpec(
        name="word_count",
        description="Count the words in a piece of text",
        input_schema={"type": "object",
                      "properties": {"text": {"type": "string"}},
                      "required": ["text"],
                      "additionalProperties": False},   # refuse anything not declared
        risk="read",                                     # no side effect -> read
    ),
    lambda text: {"words": len(text.split())},           # the actual implementation
)

print("Registered:", [s.name for s in mine.discover()])
print("Valid call:", mine.call("word_count", {"text": "a harness is a runtime plus its rules"}))

try:
    mine.call("word_count", {"text": 12345})             # a number, not a string
except TypeError as exc:
    print("Wrong type refused:", exc)

try:
    mine.register(ToolSpec("word_count", "duplicate", {"type": "object"}, "read"), lambda: None)
except ValueError as exc:
    print("Duplicate registration refused:", exc)

### Checkpoint

**1. The registry validated the arguments. Why is policy still needed?**

<details><summary>Show answer</summary>

Validation only answers *is this well-formed?*. `send_email` with a valid address and a valid body is perfectly well-formed and may still be something this agent must not do unsupervised. Shape and permission are different questions, decided by different code.

</details>

**2. Why does `additionalProperties: False` matter?**

<details><summary>Show answer</summary>

Without it, a model can attach arguments nobody declared, and they flow straight into your function call. Declaring the schema closed means an unexpected argument is a rejected call rather than a surprise keyword argument.

</details>

### Recap

- Limitation: hardcoded if/elif tool dispatch cannot give two agents different tools, and has nowhere to record a tool's risk.
- Layer added: a registry holding name, description, schema, risk and function, with per-agent scoped discovery and structural validation.
- Evidence: four tools registered, one visible to the research agent, and four different malformed calls rejected before any function body ran.
